# Q3 — 오탐 줄이기 (precision 개선)
Q1 베이스라인은 클래스 가중치가 공격적(최대 20배)이라 결함을 남발 → recall 높지만 precision 낮음.
여기서는 **class-balanced 가중치 + Focal Loss**로 바꿔 오탐을 줄이고, Q1과 직접 비교합니다.

In [ ]:
# --- 한글 폰트 + import ---
import sys
from pathlib import Path
_p = Path.cwd()
for cand in [_p, _p.parent, _p.parent.parent]:
    if (cand / "utils").is_dir():
        sys.path.insert(0, str(cand)); PROJ = cand; break
else:
    PROJ = _p
from utils.korean_font import set_korean_font
set_korean_font()

import numpy as np, matplotlib.pyplot as plt, time
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score, precision_score, recall_score

CLASSES = ["Center","Donut","Edge-Loc","Edge-Ring","Loc","Near-full","Random","Scratch","none"]
NUM_CLASSES = len(CLASSES)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

In [ ]:
# --- 설정 ---
PROC = PROJ/"data"/"processed"
CKPT_DIR = PROJ/"models"/"checkpoints"; CKPT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR  = PROJ/"results"/"figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE=128; BATCH=256; EPOCHS=6; LR=3e-4; SEED=42
torch.manual_seed(SEED); np.random.seed(SEED)

# === 오탐 줄이기 핵심 하이퍼파라미터 ===
LOSS_MODE = "focal"      # "focal" | "weighted_ce"
FOCAL_GAMMA = 2.0        # 클수록 쉬운 샘플 무시 (오탐 억제에 도움)
CB_BETA = 0.999          # class-balanced 가중치 강도 (1에 가까울수록 강함; 0.9999=강, 0.99=약)
# 비교: Q1은 inverse-freq, cap=20 (너무 공격적 -> 오탐 많음)
print(f"LOSS_MODE={LOSS_MODE}, gamma={FOCAL_GAMMA}, cb_beta={CB_BETA}")

In [ ]:
# --- 데이터 로드 + Dataset ---
def load_split(n):
    d = np.load(PROC/f"wafer_{n}.npz", allow_pickle=True); return d["X"], d["y"]
Xtr,ytr = load_split("train"); Xva,yva = load_split("val"); Xte,yte = load_split("test")
counts = np.bincount(ytr, minlength=NUM_CLASSES)
print("train:", Xtr.shape, "| counts:", counts.tolist())

class WaferDS(Dataset):
    def __init__(self, X, y, train=False): self.X=X; self.y=y; self.train=train
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        img = self.X[i].astype(np.float32)/2.0
        if self.train:
            k=np.random.randint(0,4)
            if k: img=np.rot90(img,k).copy()
            if np.random.rand()<0.5: img=np.fliplr(img).copy()
            if np.random.rand()<0.5: img=np.flipud(img).copy()
        img=(img-0.5)/0.5
        img=np.expand_dims(img,0).repeat(3,0)
        return torch.from_numpy(img), int(self.y[i])

mk = lambda X,y,tr: DataLoader(WaferDS(X,y,tr), batch_size=BATCH, shuffle=tr, num_workers=0, pin_memory=(DEVICE=="cuda"))
tr_loader=mk(Xtr,ytr,True); va_loader=mk(Xva,yva,False); te_loader=mk(Xte,yte,False)

In [ ]:
# --- class-balanced 가중치 (effective number of samples) ---
# w_c = (1 - beta) / (1 - beta^n_c)  -> inverse-freq보다 훨씬 부드러움
eff = 1.0 - np.power(CB_BETA, counts)
cb_w = (1.0 - CB_BETA) / np.maximum(eff, 1e-12)
cb_w = cb_w / cb_w.sum() * NUM_CLASSES      # 평균 1 정도로 정규화
cb_w = cb_w.astype(np.float32)
class_w = torch.tensor(cb_w, device=DEVICE)
print("CB 가중치:", {c: round(float(x),2) for c,x in zip(CLASSES, cb_w)})
print("(참고) Q1 inverse-freq cap20 대비 최대값이 훨씬 작음 -> 결함 남발 줄어듦")

In [ ]:
# --- Focal Loss (가중치 포함) ---
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super().__init__(); self.weight=weight; self.gamma=gamma
    def forward(self, logits, target):
        logp = F.log_softmax(logits, dim=1)
        p = logp.exp()
        ce = F.nll_loss(logp, target, weight=self.weight, reduction="none")
        pt = p.gather(1, target.unsqueeze(1)).squeeze(1)
        return ((1-pt)**self.gamma * ce).mean()

if LOSS_MODE == "focal":
    criterion = FocalLoss(weight=class_w, gamma=FOCAL_GAMMA)
else:
    criterion = nn.CrossEntropyLoss(weight=class_w)
print("loss:", type(criterion).__name__)

In [ ]:
# --- 모델 + 옵티마이저 ---
try: net = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
except Exception: net = models.resnet18(pretrained=True)
net.fc = nn.Linear(net.fc.in_features, NUM_CLASSES); net=net.to(DEVICE)
optimizer = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
# --- 학습 (val macro-F1 best 저장) ---
@torch.no_grad()
def evaluate(loader):
    net.eval(); ys=[]; ps=[]
    for xb,yb in loader:
        out=net(xb.to(DEVICE)); ps.append(out.argmax(1).cpu().numpy()); ys.append(yb.numpy())
    y=np.concatenate(ys); p=np.concatenate(ps)
    return accuracy_score(y,p), f1_score(y,p,average="macro"), y, p

best_f1=-1; best_path=CKPT_DIR/"q3_focal_resnet18.pt"; hist={"loss":[],"val_f1":[]}
for ep in range(1,EPOCHS+1):
    net.train(); t0=time.time(); run=0.0
    for bi,(xb,yb) in enumerate(tr_loader):
        xb,yb=xb.to(DEVICE),yb.to(DEVICE)
        optimizer.zero_grad(); loss=criterion(net(xb),yb); loss.backward(); optimizer.step()
        run+=loss.item()
        if bi%100==0: print(f"  ep{ep} {bi}/{len(tr_loader)} loss={loss.item():.3f}", end="\r")
    scheduler.step()
    va_acc,va_f1,_,_=evaluate(va_loader); hist["loss"].append(run/len(tr_loader)); hist["val_f1"].append(va_f1)
    flag=""
    if va_f1>best_f1: best_f1=va_f1; torch.save({"model":net.state_dict(),"classes":CLASSES}, best_path); flag="  <- best"
    print(f"[ep{ep}/{EPOCHS}] loss={run/len(tr_loader):.3f} | val_macroF1={va_f1:.3f} | {time.time()-t0:.0f}s{flag}")
print("\nbest val macro-F1:", round(best_f1,4))

In [ ]:
# --- test 평가 + Q1 베이스라인과 비교 ---
ck=torch.load(best_path, map_location=DEVICE); net.load_state_dict(ck["model"])
te_acc,te_f1,yt,pt=evaluate(te_loader)
prec=precision_score(yt,pt,average=None,zero_division=0)
rec =recall_score(yt,pt,average=None,zero_division=0)

# Q1 베이스라인 수치 (앞 실행 결과 기록)
q1_prec={"Center":0.726,"Donut":0.811,"Edge-Loc":0.596,"Edge-Ring":0.957,"Loc":0.558,
         "Near-full":0.815,"Random":0.688,"Scratch":0.424,"none":0.998}
print(f"TEST  accuracy={te_acc:.3f} | macro-F1={te_f1:.3f}  (Q1: acc=0.943, F1=0.811)\n")
print(f"{'class':10s} {'prec_Q1':>8s} {'prec_Q3':>8s} {'Δprec':>7s} {'recall':>7s}")
for i,c in enumerate(CLASSES):
    dp=prec[i]-q1_prec[c]
    print(f"{c:10s} {q1_prec[c]:8.3f} {prec[i]:8.3f} {dp:+7.3f} {rec[i]:7.3f}")
print("\n(+면 오탐 줄어 precision 개선. recall 너무 떨어지면 gamma/beta 조절)")

In [ ]:
# --- 혼동행렬 ---
import itertools
cm=confusion_matrix(yt,pt); cmn=cm/cm.sum(1,keepdims=True).clip(min=1)
fig,ax=plt.subplots(figsize=(8,7)); im=ax.imshow(cmn,cmap="Blues",vmin=0,vmax=1)
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASSES,rotation=45,ha="right")
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASSES)
ax.set_xlabel("예측"); ax.set_ylabel("실제"); ax.set_title("Q3 정규화 혼동행렬")
for i,j in itertools.product(range(NUM_CLASSES),range(NUM_CLASSES)):
    ax.text(j,i,f"{cmn[i,j]:.2f}",ha="center",va="center",color="white" if cmn[i,j]>0.5 else "black",fontsize=8)
fig.colorbar(im,fraction=0.046); plt.tight_layout()
plt.savefig(FIG_DIR/"q3_confusion.png",dpi=110,bbox_inches="tight"); plt.show()

## 해석 가이드

- **Δprec 가 +** 면 그 클래스 오탐(헛알람)이 줄어든 것 → 목표 달성.
- recall 이 너무 떨어지면(놓침 증가) `FOCAL_GAMMA`를 1.0~1.5로 낮추거나 `CB_BETA`를 0.99로 줄여 가중치를 더 부드럽게.
- 반대로 오탐이 여전히 많으면 `CB_BETA`를 0.9999로 올리거나 gamma를 3.0으로.

**튜닝 사이클**: 설정 셀 값만 바꾸고 → 커널 재시작 없이 모델 셀부터 다시 실행 → 비교표 확인. 1~2번 돌려보면 precision/recall 균형점이 잡혀요.

다음 후보: 균형점 찾으면 Streamlit 데모로 마무리하거나, 결함 원인 멀티태스크(Phase 2)로 확장.